# EDA: Twitter Sentiment Analysis (Hate Speech) - Варіант 19

Первинний аналіз даних для виявлення расистських/сексистських твітів.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path("../data/raw")
data_file = DATA_PATH / "train.csv"
if not data_file.exists():
    data_file = Path("../sample_data/train_sample.csv")
df = pd.read_csv(data_file)
df.head()

## Типи даних та пропуски

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## Розподіл цільової змінної (Target Distribution)

In [ ]:
label_col = 'label' if 'label' in df.columns else 'Label'
df[label_col].value_counts()

In [ ]:
df[label_col].value_counts().plot(kind='bar')
plt.title('Розподіл класів (0: non-hate, 1: hate)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Довжина твітів

In [ ]:
text_col = 'tweet' if 'tweet' in df.columns else 'Tweet'
df['word_count'] = df[text_col].astype(str).str.split().str.len()
df['char_count'] = df[text_col].astype(str).str.len()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['word_count'].hist(ax=axes[0], bins=50)
axes[0].set_title('Розподіл кількості слів на твіт')
axes[0].set_xlabel('Кількість слів')
df['char_count'].hist(ax=axes[1], bins=50)
axes[1].set_title('Розподіл кількості символів на твіт')
axes[1].set_xlabel('Кількість символів')
plt.tight_layout()
plt.show()

## Частотність слів по класах

In [ ]:
from collections import Counter

def get_top_words(series, n=20):
    all_words = []
    for text in series.astype(str):
        all_words.extend(text.lower().split())
    return Counter(all_words).most_common(n)

hate_tweets = df[df[label_col] == 1][text_col]
non_hate_tweets = df[df[label_col] == 0][text_col]

top_hate = get_top_words(hate_tweets)
top_non_hate = get_top_words(non_hate_tweets)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
words_hate, counts_hate = zip(*top_hate)
axes[0].barh(range(len(words_hate)), counts_hate)
axes[0].set_yticks(range(len(words_hate)))
axes[0].set_yticklabels(words_hate)
axes[0].invert_yaxis()
axes[0].set_title('Топ-20 слів у hate speech твітах')

words_non, counts_non = zip(*top_non_hate)
axes[1].barh(range(len(words_non)), counts_non)
axes[1].set_yticks(range(len(words_non)))
axes[1].set_yticklabels(words_non)
axes[1].invert_yaxis()
axes[1].set_title('Топ-20 слів у non-hate твітах')
plt.tight_layout()
plt.show()

## Приклади твітів

In [ ]:
print('=== Приклади HATE (label=1) ===')
for i, row in df[df[label_col] == 1].head(5).iterrows():
    print(row[text_col][:100])
    print('---')

print('\n=== Приклади NON-HATE (label=0) ===')
for i, row in df[df[label_col] == 0].head(5).iterrows():
    print(row[text_col][:100])
    print('---')